# Tutorial 2 – Accessing ARCEME Cubes for Footprint Analysis

This notebook demonstrates how to open, explore, and analyse ARCEME data cubes
stored as Zarr archives on public CloudFerro object storage. No credentials are needed.

**Workflows covered:**

1. Opening a Zarr cube with `xarray`
2. Inspecting structure and metadata
3. Visualising Sentinel-2 RGB composites across time
4. Applying the cloud mask to filter observations
5. Computing NDVI and NDVI anomaly around the event date
6. Overlaying Sentinel-1 SAR backscatter for multi-source analysis

---
**Requirements**
```
pip install xarray zarr numpy matplotlib pandas pyproj
```
*(all included in the project environment via `uv sync`)*

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
from datetime import datetime, timedelta

## 1. Opening a Zarr cube

The ARCEME cubes are publicly accessible over HTTPS. Simply pass the URL to
`xarray.open_zarr` — no authentication required.

In [ ]:
BASE_URL = "https://s3.waw3-2.cloudferro.com/swift/v1/ARCEME-DC-TEST"

# Available cubes in the collection
import requests
listing = requests.get(f"{BASE_URL}/?delimiter=/&format=json").json()
cube_names = sorted(
    e["subdir"].rstrip("/")
    for e in listing if "subdir" in e and e["subdir"].endswith(".zarr/")
)
print(f"Available cubes ({len(cube_names)}):")
for name in cube_names:
    print(" ", name)

In [ ]:
# Select one cube to analyse
# Format: DC__wocat_{wocat_id}_dhp_{dhp_id}__{start}__{end}.zarr
CUBE_NAME = "DC__wocat_1007_dhp_28028__2017-10-27__2019-10-27.zarr"

CUBE_URL = f"{BASE_URL}/{CUBE_NAME}"
print("Loading:", CUBE_URL)

ds = xr.open_zarr(CUBE_URL, consolidated=True)
ds

## 2. Inspecting structure and metadata

In [ ]:
print("=== Dimensions ===")
for dim, size in ds.sizes.items():
    print(f"  {dim}: {size}")

print("\n=== Data variables ===")
for var in ds.data_vars:
    dims = list(ds[var].dims)
    print(f"  {var:<15}  dims={dims}  dtype={ds[var].dtype}")

print("\n=== Key metadata ===")
meta_keys = ["project", "time_coverage_start", "time_coverage_end",
             "epsg", "pixel_resolution_meters", "tile_edge_size_meters",
             "temporal_window", "cloud_mask_algorithm"]
for k in meta_keys:
    if k in ds.attrs:
        print(f"  {k}: {ds.attrs[k]}")

In [ ]:
# Derive event date from metadata (midpoint of the ±12 month window)
t_start = pd.Timestamp(ds.attrs["time_coverage_start"])
t_end   = pd.Timestamp(ds.attrs["time_coverage_end"])
event_date = t_start + (t_end - t_start) / 2

print(f"Window start : {t_start.date()}")
print(f"Event date   : {event_date.date()}")
print(f"Window end   : {t_end.date()}")

# Convenience: time arrays as pandas DatetimeIndex
t_s2 = pd.DatetimeIndex(ds["time_sentinel_2_l2a"].values)
t_s1 = pd.DatetimeIndex(ds["time_sentinel_1_rtc"].values)
print(f"\nS2 observations : {len(t_s2)}")
print(f"S1 observations : {len(t_s1)}")

## 3. Sentinel-2 RGB composites

Bands are stored as `uint16` (scale factor = 10 000).  
We build a true-colour composite (B04 = red, B03 = green, B02 = blue).

In [ ]:
SCALE = 10_000.0   # Sentinel-2 L2A quantification value

def make_rgb(ds, time_idx, brightness=3.5, percentile=2):
    """Return an (H, W, 3) float array clipped to [0, 1] for display."""
    sel = {"time_sentinel_2_l2a": time_idx}
    r = ds["B04"].isel(**sel).values.astype(float) / SCALE
    g = ds["B03"].isel(**sel).values.astype(float) / SCALE
    b = ds["B02"].isel(**sel).values.astype(float) / SCALE
    rgb = np.stack([r, g, b], axis=-1)
    # Stretch using percentile clipping
    lo = np.nanpercentile(rgb, percentile)
    hi = np.nanpercentile(rgb, 100 - percentile)
    rgb = np.clip((rgb - lo) / (hi - lo + 1e-9) * brightness, 0, 1)
    return rgb

In [ ]:
# Show 6 evenly spaced S2 acquisitions across the full time window
n_panels = 6
indices = np.linspace(0, len(t_s2) - 1, n_panels, dtype=int)

fig, axes = plt.subplots(1, n_panels, figsize=(3 * n_panels, 3.5))
for ax, idx in zip(axes, indices):
    rgb = make_rgb(ds, idx)
    ax.imshow(rgb, origin="upper")
    date_str = t_s2[idx].strftime("%Y-%m-%d")
    label = "[EVENT]" if abs((t_s2[idx] - event_date).days) < 20 else ""
    ax.set_title(f"{date_str}\n{label}", fontsize=9)
    ax.axis("off")

fig.suptitle("Sentinel-2 True Colour (B04/B03/B02)", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 4. Applying the cloud mask

The `cloud_mask` variable uses the SEnSeIv2/SegFormerB2 model with four classes:

| Value | Class |
|---|---|
| 0 | Unoccluded (clear) |
| 1 | Thick cloud |
| 2 | Thin cloud |
| 3 | Shadow |

In [ ]:
# Compute fraction of clear pixels per acquisition
cloud_mask = ds["cloud_mask"].astype(float)   # (time_s2, y, x), uint16
clear_frac = (
    (cloud_mask == 0)
    .mean(dim=["y", "x"])
    .compute()
    .values
)

fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(t_s2, clear_frac * 100, width=2.5, color="steelblue", alpha=0.85)
ax.axvline(event_date, color="red", lw=1.5, ls="--", label="Event date")
ax.set_ylim(0, 100)
ax.set_ylabel("Clear pixels (%)")
ax.set_title("Cloud-free coverage per Sentinel-2 acquisition")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
ax.legend()
plt.tight_layout()
plt.show()

# Keep only acquisitions with >50% clear pixels
CLEAR_THRESHOLD = 0.50
clear_idx = np.where(clear_frac >= CLEAR_THRESHOLD)[0]
print(f"Acquisitions with ≥{CLEAR_THRESHOLD*100:.0f}% clear pixels: "
      f"{len(clear_idx)} / {len(t_s2)}")

In [ ]:
def make_rgb_masked(ds, time_idx):
    """True-colour composite with cloudy/shadow pixels set to NaN (white in display)."""
    sel = {"time_sentinel_2_l2a": time_idx}
    mask = ds["cloud_mask"].isel(**sel).values  # 0 = clear
    r = ds["B04"].isel(**sel).values.astype(float) / SCALE
    g = ds["B03"].isel(**sel).values.astype(float) / SCALE
    b = ds["B02"].isel(**sel).values.astype(float) / SCALE
    for band in (r, g, b):
        band[mask != 0] = np.nan   # mask clouds + shadows
    rgb = np.stack([r, g, b], axis=-1)
    valid = ~np.isnan(rgb).any(axis=-1)
    lo = np.nanpercentile(rgb[valid], 2)
    hi = np.nanpercentile(rgb[valid], 98)
    rgb = np.clip((rgb - lo) / (hi - lo + 1e-9) * 3.5, 0, 1)
    return rgb

# Show the 4 clearest acquisitions after masking
top_idx = clear_idx[np.argsort(clear_frac[clear_idx])[::-1][:4]]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, idx in zip(axes, top_idx):
    rgb = make_rgb_masked(ds, idx)
    ax.imshow(rgb, origin="upper")
    pre_post = "pre-event" if t_s2[idx] < event_date else "post-event"
    ax.set_title(f"{t_s2[idx].strftime('%Y-%m-%d')}\n({pre_post}, {clear_frac[idx]*100:.0f}% clear)",
                 fontsize=9)
    ax.axis("off")

fig.suptitle("Sentinel-2 RGB – cloud-masked (white = masked)", fontsize=12)
plt.tight_layout()
plt.show()

## 5. NDVI anomaly around the event date

We compute the Normalised Difference Vegetation Index (NDVI) for every
clear acquisition, then measure the departure from the pre-event baseline
to detect vegetation change linked to the extreme event.

$$\text{NDVI} = \frac{\text{B08} - \text{B04}}{\text{B08} + \text{B04}}$$

$$\text{NDVI anomaly} = \text{NDVI}(t) - \overline{\text{NDVI}}_{\text{pre-event}}$$

In [ ]:
# Compute NDVI for all clear acquisitions (lazy, per-pixel)
clear_times = ds["time_sentinel_2_l2a"].isel(time_sentinel_2_l2a=clear_idx)
b04 = ds["B04"].sel(time_sentinel_2_l2a=clear_times).astype(float) / SCALE
b08 = ds["B08"].sel(time_sentinel_2_l2a=clear_times).astype(float) / SCALE

# Apply cloud mask: set occluded pixels to NaN
mask_clear = (ds["cloud_mask"].sel(time_sentinel_2_l2a=clear_times) == 0)
b04 = b04.where(mask_clear)
b08 = b08.where(mask_clear)

ndvi = (b08 - b04) / (b08 + b04 + 1e-9)
ndvi = ndvi.rename({"time_sentinel_2_l2a": "time"})
ndvi = ndvi.assign_coords(time=("time", clear_times.values))

print(f"NDVI computed for {len(clear_idx)} clear acquisitions")

In [ ]:
# Separate pre- and post-event periods
t_clear = pd.DatetimeIndex(clear_times.values)
pre_mask  = t_clear < event_date
post_mask = t_clear >= event_date

# Mean NDVI baseline from pre-event observations
ndvi_pre_mean  = ndvi.isel(time=np.where(pre_mask)[0]).mean(dim="time").compute()

# Anomaly = deviation from pre-event mean
ndvi_post      = ndvi.isel(time=np.where(post_mask)[0])
ndvi_anomaly   = (ndvi_post - ndvi_pre_mean).compute()

print(f"Pre-event acquisitions  : {pre_mask.sum()}")
print(f"Post-event acquisitions : {post_mask.sum()}")
print(f"Mean pre-event NDVI     : {float(ndvi_pre_mean.mean()):.3f}")

In [ ]:
# Spatial mean NDVI over time (timeline plot)
ndvi_ts = ndvi.mean(dim=["y", "x"]).compute().values

fig, ax = plt.subplots(figsize=(14, 4))
pre_color  = "steelblue"
post_color = "darkorange"
ax.scatter(t_clear[pre_mask],  ndvi_ts[pre_mask],  c=pre_color,  s=35, label="Pre-event",  zorder=3)
ax.scatter(t_clear[post_mask], ndvi_ts[post_mask], c=post_color, s=35, label="Post-event", zorder=3)
ax.axhline(float(ndvi_pre_mean.mean()), color=pre_color, ls="--", alpha=0.5,
           label=f"Pre-event mean NDVI ({float(ndvi_pre_mean.mean()):.3f})")
ax.axvline(event_date, color="red", lw=1.5, ls="-", label="Event date")
ax.set_ylabel("Mean NDVI (spatial average)")
ax.set_title("NDVI time series – clear observations only")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Spatial NDVI anomaly maps: mean over first 3 and last 3 post-event observations
n_show = min(3, len(np.where(post_mask)[0]))
post_indices = np.where(post_mask)[0]

fig, axes = plt.subplots(2, n_show, figsize=(4.5 * n_show, 8))
vmax = 0.3
cmap = plt.cm.RdYlGn

for col, p_idx in enumerate(post_indices[:n_show]):
    local_idx = col   # index within post-event ndvi_anomaly
    anom = ndvi_anomaly.isel(time=local_idx).values
    date_str = t_clear[post_indices[col]].strftime("%Y-%m-%d")

    # NDVI anomaly map
    im = axes[0, col].imshow(anom, cmap=cmap, vmin=-vmax, vmax=vmax, origin="upper")
    axes[0, col].set_title(f"NDVI anomaly\n{date_str}", fontsize=9)
    axes[0, col].axis("off")
    plt.colorbar(im, ax=axes[0, col], fraction=0.04, pad=0.02)

    # Corresponding RGB
    # Map post_indices[col] back to global clear_idx
    global_idx = clear_idx[post_indices[col]]
    rgb = make_rgb_masked(ds, global_idx)
    axes[1, col].imshow(rgb, origin="upper")
    axes[1, col].set_title(f"RGB (cloud-masked)\n{date_str}", fontsize=9)
    axes[1, col].axis("off")

fig.suptitle("Post-event NDVI anomaly vs pre-event mean  |  green = gain, red = loss",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## 6. Sentinel-1 SAR backscatter – multi-source analysis

The cube includes co-polarised (VV) and cross-polarised (VH) SAR backscatter
from Sentinel-1 RTC. The VH/VV ratio is sensitive to vegetation structure
and soil moisture changes linked to extreme events.

In [ ]:
t_s1_pd = pd.DatetimeIndex(ds["time_sentinel_1_rtc"].values)
print(f"S1 acquisitions: {len(t_s1_pd)} "
      f"({t_s1_pd.min().date()} → {t_s1_pd.max().date()})")

# Orbit pass direction (ascending / descending)
if "orbit_state" in ds.coords:
    asc  = (ds["orbit_state"] == "ascending").sum().item()
    desc = (ds["orbit_state"] == "descending").sum().item()
    print(f"  Ascending passes : {asc}")
    print(f"  Descending passes: {desc}")

In [ ]:
# Compute monthly median VH backscatter (dB) and track change over time
vh_db = 10 * np.log10(ds["vh"].where(ds["vh"] > 0) + 1e-12)  # linear → dB

# Spatial mean per acquisition (lazy)
vh_mean = vh_db.mean(dim=["y", "x"]).compute().values
vv_mean = (10 * np.log10(ds["vv"].where(ds["vv"] > 0) + 1e-12)
           .mean(dim=["y", "x"]).compute().values)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for ax, values, label, color in [
    (axes[0], vv_mean, "VV (dB)", "steelblue"),
    (axes[1], vh_mean, "VH (dB)", "darkorange"),
]:
    pre  = t_s1_pd < event_date
    post = t_s1_pd >= event_date
    ax.scatter(t_s1_pd[pre],  values[pre],  color=color, s=15, alpha=0.7, label="Pre-event")
    ax.scatter(t_s1_pd[post], values[post], color=color, s=15, alpha=0.7,
               marker="^", label="Post-event")
    ax.axhline(np.median(values[pre]),  color=color, ls="--", alpha=0.5,
               label=f"Pre-event median ({np.median(values[pre]):.1f} dB)")
    ax.axvline(event_date, color="red", lw=1.5, label="Event date")
    ax.set_ylabel(label)
    ax.legend(fontsize=8, ncol=4)
    ax.grid(True, alpha=0.3)

axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha="right")
fig.suptitle("Sentinel-1 RTC backscatter time series (spatial mean)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side: S1 VH map closest to event date (pre vs post)
def nearest_s1_idx(target):
    return int(np.argmin(np.abs(t_s1_pd - target)))

idx_pre  = nearest_s1_idx(event_date - pd.Timedelta(days=30))
idx_post = nearest_s1_idx(event_date + pd.Timedelta(days=30))

def s1_map(ds, time_idx, band="vh"):
    data = ds[band].isel(time_sentinel_1_rtc=time_idx).values.astype(float)
    data_db = 10 * np.log10(np.where(data > 0, data, np.nan))
    return data_db

vmin_db, vmax_db = -25, -5   # typical S1 VH range over vegetated land

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, idx, title in [
    (axes[0], idx_pre,  f"VH pre-event\n{t_s1_pd[idx_pre].strftime('%Y-%m-%d')}"),
    (axes[1], idx_post, f"VH post-event\n{t_s1_pd[idx_post].strftime('%Y-%m-%d')}"),
]:
    im = ax.imshow(s1_map(ds, idx), cmap="Greys_r",
                   vmin=vmin_db, vmax=vmax_db, origin="upper")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.04, label="dB")

# Difference map: post − pre
diff = s1_map(ds, idx_post) - s1_map(ds, idx_pre)
im3 = axes[2].imshow(diff, cmap="bwr", vmin=-5, vmax=5, origin="upper")
axes[2].set_title("VH difference (post − pre) [dB]", fontsize=10)
axes[2].axis("off")
plt.colorbar(im3, ax=axes[2], fraction=0.04, label="ΔdB")

fig.suptitle("Sentinel-1 VH backscatter around event date", fontsize=12)
plt.tight_layout()
plt.show()

## 7. Combined multi-source overview

A single figure combining RGB composite, NDVI anomaly, SAR VH difference
and ESA WorldCover land cover for the event area.

In [ ]:
# Pick clearest post-event S2 scene
best_post = clear_idx[post_indices[int(np.argmax(clear_frac[post_indices]))]]
best_anom_idx = int(np.argmax(clear_frac[post_indices]))

# Land cover (single time step)
lc = ds["ESA_LC"].isel(time_esa_worldcover=0).values

# NDVI anomaly for the best post-event scene
best_anom = ndvi_anomaly.isel(time=min(best_anom_idx, len(ndvi_anomaly.time) - 1)).values

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# Panel 1: RGB
axes[0].imshow(make_rgb_masked(ds, best_post), origin="upper")
axes[0].set_title(f"Sentinel-2 RGB\n{t_s2[best_post].strftime('%Y-%m-%d')}", fontsize=10)
axes[0].axis("off")

# Panel 2: NDVI anomaly
im2 = axes[1].imshow(best_anom, cmap="RdYlGn", vmin=-0.3, vmax=0.3, origin="upper")
axes[1].set_title("NDVI anomaly\n(post − pre-event mean)", fontsize=10)
axes[1].axis("off")
plt.colorbar(im2, ax=axes[1], fraction=0.04)

# Panel 3: SAR VH difference
im3 = axes[2].imshow(diff, cmap="bwr", vmin=-5, vmax=5, origin="upper")
axes[2].set_title("Sentinel-1 VH diff\n(post − pre) [dB]", fontsize=10)
axes[2].axis("off")
plt.colorbar(im3, ax=axes[2], fraction=0.04)

# Panel 4: ESA WorldCover
lc_cmap = mcolors.ListedColormap([
    "#006400",  # 10 Tree cover
    "#ffbb22",  # 20 Shrubland
    "#ffff4c",  # 30 Grassland
    "#f096ff",  # 40 Cropland
    "#fa0000",  # 50 Built-up
    "#b4b4b4",  # 60 Bare/sparse veg
    "#f0f0f0",  # 70 Snow/ice
    "#0064c8",  # 80 Permanent water
    "#0096a0",  # 90 Herbaceous wetland
    "#00cf75",  # 95 Mangroves
    "#fae6a0",  # 100 Moss/lichen
])
lc_classes = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
# Map lc values to 0-based indices for the colormap
lc_display = np.full_like(lc, np.nan, dtype=float)
for i, cls in enumerate(lc_classes):
    lc_display[lc == cls] = i

im4 = axes[3].imshow(lc_display, cmap=lc_cmap, vmin=-0.5,
                     vmax=len(lc_classes) - 0.5, origin="upper")
axes[3].set_title("ESA WorldCover 2020", fontsize=10)
axes[3].axis("off")
lc_labels = ["Tree cover","Shrubland","Grassland","Cropland","Built-up",
             "Bare/sparse","Snow/ice","Water","Wetland","Mangroves","Moss"]
cbar4 = plt.colorbar(im4, ax=axes[3], fraction=0.04,
                     ticks=range(len(lc_classes)))
cbar4.set_ticklabels(lc_labels)
cbar4.ax.tick_params(labelsize=7)

fig.suptitle(
    f"Multi-source overview – event date: {event_date.strftime('%Y-%m-%d')}",
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.show()

## 8. Iterating over multiple cubes

The same analysis can be repeated across all available cubes in the collection.

In [ ]:
import re

PATTERN = re.compile(
    r"DC__wocat_(\d+)_dhp_(\d+)__(\d{4}-\d{2}-\d{2})__(\d{4}-\d{2}-\d{2})\.zarr"
)

results = []
for name in cube_names:
    m = PATTERN.match(name)
    if not m:
        continue
    wocat_id, dhp_id, t0_str, t1_str = m.groups()
    results.append({
        "wocat_id":   wocat_id,
        "dhp_id":     dhp_id,
        "start_date": t0_str,
        "end_date":   t1_str,
        "event_date": str((pd.Timestamp(t0_str) +
                           (pd.Timestamp(t1_str) - pd.Timestamp(t0_str)) / 2).date()),
        "url":        f"{BASE_URL}/{name}",
    })

df_cubes = pd.DataFrame(results)
print(f"Collection summary ({len(df_cubes)} cubes):")
df_cubes

In [ ]:
# Example: compute mean NDVI anomaly (post − pre) for every cube without
# loading all spatial data – only the temporal mean is pulled.
# Uncomment to run (requires fetching data for all 20 cubes over HTTP).

# summaries = []
# for _, row in df_cubes.iterrows():
#     try:
#         ds_i = xr.open_zarr(row["url"], consolidated=True)
#         ev   = pd.Timestamp(row["event_date"])
#         t_i  = pd.DatetimeIndex(ds_i["time_sentinel_2_l2a"].values)
#         cm_i = ds_i["cloud_mask"]
#         clear_i = (cm_i == 0).mean(dim=["y", "x"]).compute().values >= 0.5
#         t_clear_i = t_i[clear_i]
#         b4 = ds_i["B04"].sel(time_sentinel_2_l2a=t_clear_i).astype(float) / SCALE
#         b8 = ds_i["B08"].sel(time_sentinel_2_l2a=t_clear_i).astype(float) / SCALE
#         ndvi_i = ((b8 - b4) / (b8 + b4 + 1e-9)).mean(dim=["y", "x"]).compute().values
#         pre  = t_clear_i < ev
#         post = t_clear_i >= ev
#         delta = ndvi_i[post].mean() - ndvi_i[pre].mean() if pre.any() and post.any() else np.nan
#         summaries.append({"wocat_id": row["wocat_id"], "ndvi_anomaly_mean": delta})
#         print(f"  {row['wocat_id']}  ΔNDVI={delta:+.3f}")
#     except Exception as e:
#         print(f"  {row['wocat_id']}  ERROR: {e}")
#
# pd.DataFrame(summaries)

## Summary

In this tutorial you learned how to:

| Step | Method |
|---|---|
| Open a cube | `xr.open_zarr(url, consolidated=True)` |
| Filter cloudy pixels | `cloud_mask == 0` (SEnSeIv2/SegFormerB2) |
| Compute NDVI | `(B08 − B04) / (B08 + B04)` using scaled uint16 bands |
| Detect change | NDVI anomaly = post-event NDVI − pre-event mean |
| Analyse SAR | VH/VV backscatter in dB; difference map post − pre |
| Contextualise | ESA WorldCover 2020 land cover layer |

The ARCEME collection is linked from the
[ESA Open Science Data Catalogue (ARCEME entry)](https://opensciencedata.esa.int/).